# Модуль 3 · Анализ изменений СПР

**Путь данных:** JSON/XML → плоские строки → сравнение → качество → разрезы → блоки → факты → проверки → сводка.

Этот ноутбук вызывает функции работающего `engine/`. Здесь находятся только
порядок вызовов, пояснения и отображение результатов. Алгоритмы сравнения,
бакетизации, агрегации и проверок не переписаны в ячейках.

**По умолчанию — четыре публичные пары из samples, а не полный датасет.**
Для своего датасета укажите папку с `manifest.json` и `RESULTS` в `DATASET_PATH`.
После изменения параметров: **Kernel → Restart Kernel and Run All Cells**.

В каждой стадии можно раскрыть «Логика и исходный код». Новые результаты пишутся
в отдельный каталог `runs/notebook_<время>_<id>`; прошлые запуски не перезаписываются.

In [1]:
from pathlib import Path
import os
import sys

# Работает при открытии Jupyter из корня репозитория или notebooks/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "engine/pipeline.py").is_file() and
                (p / "configs/default_spr.yaml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Запустите Jupyter из корня репозитория или notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

### Параметры запуска

Оставьте `DATASET_PATH` пустым для готовой демонстрации. Для полного прогона
впишите путь вида `r"D:/.../data/RUN-20260827-0001"`.
`VERIFY_PIPELINE=True` **повторно рассчитывает датасет** обычным пайплайном и
сравнивает результаты; время этой сверки показывается отдельно.
`RUN_LLM=False` оставляет необязательный сетевой шаг выключенным.

In [2]:
DATASET_PATH = os.environ.get("SPR_DATASET_PATH", "")  # Вставьте свой путь сюда.
CONFIG_PATH = os.environ.get("SPR_CONFIG_PATH", "configs/default_spr.yaml")
OUTPUT_ROOT = os.environ.get("SPR_NOTEBOOK_OUTPUT_ROOT", "runs")
VERIFY_PIPELINE = os.environ.get("SPR_NOTEBOOK_VERIFY_PIPELINE", "1") != "0"
RUN_LLM = False
STARTED_BY = os.environ.get("SPR_STARTED_BY", "notebook-user")

In [3]:
from contextlib import contextmanager
from datetime import datetime, timezone
from time import perf_counter
from uuid import uuid4
from copy import deepcopy
import html
import inspect
import json
import pandas as pd
import yaml
from IPython.display import display, HTML, Markdown, Code
from engine import steps
from engine.aggregates import blocks, facts
from engine.checks_runner import run_checks
from engine.mart_api import MartAPI
from engine.summary import prepare_summary
from engine.report_html import write_report_html
from engine.pipeline import run_pipeline

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 70)

def local_path(value):
    p = Path(value).expanduser()
    return p if p.is_absolute() else ROOT / p

def display_path(path):
    path = Path(path)
    return str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else path.name

config_file = local_path(CONFIG_PATH)
with config_file.open(encoding="utf-8") as stream:
    cfg = yaml.safe_load(stream) or {}
cfg = deepcopy(cfg)
using_demo = not bool(DATASET_PATH.strip())
dataset = ROOT / "notebooks/demo_data" if using_demo else local_path(DATASET_PATH)
if not dataset.is_dir():
    raise FileNotFoundError(f"Каталог данных не найден: {dataset}")
cfg.setdefault("dataset", {})["path"] = str(dataset)
# То же объединение исключений manifest, что использует prepare_comparison.
manifest_path = dataset / "manifest.json"
if cfg["dataset"].get("use_manifest_excludes", True) and manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
    excludes = cfg.setdefault("flatten", {}).setdefault("exclude_paths", [])
    excludes.extend(p for p in manifest.get("technical_fields_always_differ", []) if p not in excludes)

output_root = local_path(OUTPUT_ROOT)
run_id = "notebook_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
run_dir = output_root / run_id
run_dir.mkdir(parents=True, exist_ok=False)
TIMINGS = []

def show_source(*functions):
    for function in functions:
        title = f"{function.__module__}.{function.__name__}"
        description = inspect.getdoc(function) or "Описание см. в пояснении стадии."
        display(HTML("<details><summary><b>Логика и исходный код: " + html.escape(title) +
                     "</b></summary><pre>" + html.escape(description) + "</pre><pre>" +
                     html.escape(inspect.getsource(function)) + "</pre></details>"))

@contextmanager
def stage(name, rows_in=None):
    entry = {"step": name, "started_by": STARTED_BY,
             "started_at": datetime.now(timezone.utc).isoformat(),
             "rows_in": rows_in, "rows_out": None, "status": "running"}
    started = perf_counter()
    try:
        yield entry
        entry["status"] = "done"
    except Exception:
        entry["status"] = "error"
        raise
    finally:
        entry["finished_at"] = datetime.now(timezone.utc).isoformat()
        entry["duration_seconds"] = round(perf_counter() - started, 4)
        TIMINGS.append(entry)
        print(f"{name}: {entry['status']}; {entry['duration_seconds']:.4f} с; строк на выходе: {entry['rows_out']}")

def save_json(name, value):
    (run_dir / name).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

def preview(frame, limit=5):
    print(f"Строк: {len(frame):,}; колонок: {len(frame.columns)}")
    display(frame.head(limit))

print("Режим:", "ДЕМО — публичные samples" if using_demo else "СВОЙ ДАТАСЕТ")
print("Конфигурация:", display_path(config_file))
print("Результаты:", display_path(run_dir))
display(pd.DataFrame([{"pool": cfg.get("pool", {}).get("mode", "all"),
                      "formats": cfg["dataset"].get("formats"),
                      "max_depth": cfg.get("flatten", {}).get("max_depth"),
                      "exclude_paths": cfg.get("flatten", {}).get("exclude_paths"),
                      "checks": cfg.get("checks", [])}]))

Режим: ДЕМО — публичные samples
Конфигурация: configs/default_spr.yaml
Результаты: runs/notebook_20261107_164621_5f6d09ff


,pool,formats,max_depth,exclude_paths,checks
0,all,"[json, xml]",12,"[appData.request_id, appData.createDateTime]","[ar_drop_by_product, attribute_new_loss, external_score_drift, sto..."


## s1 · Поиск пар

**Вход:** конфигурация каталогов и пула. **Выход:** пары путей по APP_ID.
Имя файла задаёт заявку. Заявка только в одном срезе остаётся в списке;
её отсутствие будет видно в качестве данных. `pool.filter` текущий движок
пока не поддерживает: используйте `all` или `list`.

In [4]:
show_source(steps.discover)
with stage("s1_discover") as log:
    pairs = steps.discover(cfg["dataset"], cfg.get("pool"))
    log["rows_out"] = len(pairs)
# Показываем имена файлов без длинных абсолютных путей.
preview(pd.DataFrame([{"APP_ID": p["APP_ID"],
                      "BEFORE": Path(p["file_before"]).name if p["file_before"] else None,
                      "AFTER": Path(p["file_after"]).name if p["file_after"] else None} for p in pairs]))

s1_discover: done; 0.0002 с; строк на выходе: 4
Строк: 4; колонок: 3


,APP_ID,BEFORE,AFTER
0,I32554354,I32554354.json,I32554354.json
1,I32554355,I32554355.json,I32554355.json
2,I32554358,I32554358.json,I32554358.json
3,I32554362,I32554362.json,I32554362.json


## s2 · Чтение JSON/XML

**Вход:** файлы пар. **Выход:** исходные словари и диагностика.
`None` означает отсутствующий или непрочитанный документ, `{}` — корректный пустой
объект. Ошибка чтения одной стороны исключает пару из сравнения; она остаётся
в разрезах и диагностике. Так повреждённый JSON не создаёт ложного массового удаления.

In [5]:
from engine.adapters import read_any
show_source(steps.load, read_any)
with stage("s2_load", len(pairs)) as log:
    issues, doc_pairs, skipped = [], [], []
    for number, pair in enumerate(pairs, 1):
        start = len(issues)
        before = steps.load(pair["file_before"], cfg, app_id=pair["APP_ID"], source="BEFORE", issues=issues)
        after = steps.load(pair["file_after"], cfg, app_id=pair["APP_ID"], source="AFTER", issues=issues)
        doc_pairs.append({"APP_ID": pair["APP_ID"], "BEFORE": before, "AFTER": after})
        if any(item["ISSUE"] == "PARSE_ERROR" for item in issues[start:]):
            skipped.append(pair["APP_ID"])
        if number % 250 == 0:
            print(f"Прочитано пар: {number}/{len(pairs)}")
    log["rows_out"] = len(doc_pairs)
print("Пар с ошибкой чтения:", len(skipped))
preview(pd.DataFrame(issues, columns=steps.QUALITY_COLUMNS))

s2_load: done; 0.0004 с; строк на выходе: 4
Пар с ошибкой чтения: 0
Строк: 0; колонок: 5


,APP_ID,ISSUE,SOURCE,PATH,DETAILS


## s3 · Плоское представление

**Вход:** пара словарей. **Выход:** листовые атрибуты с `PATH` и `ARRAY_KEY`.
Ключ массива выбирается согласованно для двух срезов; известные ключи заданы
конфигом. Все ключи вложенных массивов входят в идентификатор строки.
Индекс применяется к векторам, а `modelName` сохраняется в ключе предикторов.
`max_depth` и фильтры учитываются движком; предупреждения попадут в s8.

In [6]:
show_source(steps.flatten_pair, steps.flatten, steps.resolve_array_rules)
with stage("s3_flatten", len(doc_pairs)) as log:
    before_rows, after_rows = [], []
    skipped_ids = set(skipped)
    for pair in doc_pairs:
        if pair["APP_ID"] in skipped_ids:
            continue
        b, a = steps.flatten_pair(pair["BEFORE"] or {}, pair["AFTER"] or {}, cfg,
                                  app_id=pair["APP_ID"], issues=issues)
        before_rows.extend(b)
        after_rows.extend(a)
    log["rows_out"] = len(before_rows) + len(after_rows)
print("BEFORE:", len(before_rows), "AFTER:", len(after_rows))
preview(pd.DataFrame(before_rows, columns=steps.NORMAL_COLUMNS))

s3_flatten: done; 0.0372 с; строк на выходе: 1148
BEFORE: 567 AFTER: 581
Строк: 567; колонок: 11


,APP_ID,SOURCE,PREFIX,CLASS,SUFFIX,ARRAY_KEY,KEY,PATH,DEPTH,VALUE,ADJUSTED_VALUE
0,I32554354,BEFORE,appData,application_id,,,application_id,appData.application_id,2,I32554354,I32554354
1,I32554354,BEFORE,decisionResult,decision,,,decision,decisionResult.decision,2,REJECT,REJECT
2,I32554354,BEFORE,decisionResult,product_trials,,15210,product_name,decisionResult.product_trials.product_name,3,KUPI_KUPI,KUPI_KUPI
3,I32554354,BEFORE,decisionResult,product_trials,,15210,product_id,decisionResult.product_trials.product_id,3,15210,15210
4,I32554354,BEFORE,decisionResult,product_trials,,15210,tariff,decisionResult.product_trials.tariff,3,190,190


## s4 · Бакеты

**Вход:** плоские строки. **Выход:** те же строки с `ADJUSTED_VALUE`.
Границы одинаковы в обоих срезах. Бакеты нужны для группировки; признак различия
позже вычисляется **по VALUE**, поэтому изменение внутри одного интервала не теряется.
Первый интервал включает левую границу, остальные имеют вид `(a, b]`.

In [7]:
show_source(steps.bucketize)
with stage("s4_bucketize", len(before_rows) + len(after_rows)) as log:
    before_rows = steps.bucketize(before_rows, cfg)
    after_rows = steps.bucketize(after_rows, cfg)
    log["rows_out"] = len(before_rows) + len(after_rows)
preview(pd.DataFrame(before_rows, columns=steps.NORMAL_COLUMNS)[["APP_ID", "PATH", "ARRAY_KEY", "VALUE", "ADJUSTED_VALUE"]])

s4_bucketize: done; 0.0018 с; строк на выходе: 1148
Строк: 567; колонок: 5


,APP_ID,PATH,ARRAY_KEY,VALUE,ADJUSTED_VALUE
0,I32554354,appData.application_id,,I32554354,I32554354
1,I32554354,decisionResult.decision,,REJECT,REJECT
2,I32554354,decisionResult.product_trials.product_name,15210,KUPI_KUPI,KUPI_KUPI
3,I32554354,decisionResult.product_trials.product_id,15210,15210,(1K – 100K]
4,I32554354,decisionResult.product_trials.tariff,15210,190,[0 – 1K]


## s5 · Вертикальное объединение

**Вход:** два списка строк. **Выход:** `union.parquet` с колонкой `SOURCE`.
Это конкатенация BEFORE и AFTER, ещё не сопоставление атрибутов.

In [8]:
show_source(steps.union)
with stage("s5_union", len(before_rows) + len(after_rows)) as log:
    union_df = steps.union(before_rows, after_rows)
    union_df.to_parquet(run_dir / "union.parquet", index=False)
    log["rows_out"] = len(union_df)
assert len(union_df) == len(before_rows) + len(after_rows)
preview(union_df)

s5_union: done; 0.0191 с; строк на выходе: 1148
Строк: 1,148; колонок: 11


,APP_ID,SOURCE,PREFIX,CLASS,SUFFIX,ARRAY_KEY,KEY,PATH,DEPTH,VALUE,ADJUSTED_VALUE
0,I32554354,BEFORE,appData,application_id,,,application_id,appData.application_id,2,I32554354,I32554354
1,I32554354,BEFORE,decisionResult,decision,,,decision,decisionResult.decision,2,REJECT,REJECT
2,I32554354,BEFORE,decisionResult,product_trials,,15210,product_name,decisionResult.product_trials.product_name,3,KUPI_KUPI,KUPI_KUPI
3,I32554354,BEFORE,decisionResult,product_trials,,15210,product_id,decisionResult.product_trials.product_id,3,15210,(1K – 100K]
4,I32554354,BEFORE,decisionResult,product_trials,,15210,tariff,decisionResult.product_trials.tariff,3,190,[0 – 1K]


## s6 · Уникальность ключей

**Ключ:** `(SOURCE, APP_ID, PATH, ARRAY_KEY)`.
Дубли показывают неоднозначное сопоставление. На следующем шаге движок исключит
такой ключ с обеих сторон и добавит `DUPLICATE_KEY`, не размножая строки merge.

In [9]:
show_source(steps.check_keys)
with stage("s6_check_keys", len(union_df)) as log:
    duplicate_rows = steps.check_keys(union_df)
    log["rows_out"] = len(duplicate_rows)
preview(duplicate_rows)

s6_check_keys: done; 0.0014 с; строк на выходе: 0
Строк: 0; колонок: 11


,APP_ID,SOURCE,PREFIX,CLASS,SUFFIX,ARRAY_KEY,KEY,PATH,DEPTH,VALUE,ADJUSTED_VALUE


## s7 · Сравнение до и после

**Вход:** union. **Выход:** `comparison.parquet`.
Outer merge даёт SAME, VALUE_CHANGED, ATTRIBUTE_NEW, ATTRIBUTE_LOST.
Текст `NULL` — значение существующего поля; отсутствующий атрибут — настоящий
null в одной стороне Parquet. Пустая строка также сохраняется отдельно.

In [10]:
show_source(steps.compare)
with stage("s7_compare", len(union_df)) as log:
    comparison_df = steps.compare(union_df, issues=issues)
    comparison_df.to_parquet(run_dir / "comparison.parquet", index=False)
    log["rows_out"] = len(comparison_df)
assert not comparison_df.duplicated(steps.ROW_KEY).any()
display(comparison_df.groupby("CHANGE_TYPE", dropna=False).size().rename("Строк атрибутов").to_frame())
preview(comparison_df.loc[comparison_df.DIFF_FLAG.eq(1)])

s7_compare: done; 0.0219 с; строк на выходе: 595
Строк: 83; колонок: 14


,Строк атрибутов
CHANGE_TYPE,
ATTRIBUTE_LOST,14
ATTRIBUTE_NEW,28
SAME,512
VALUE_CHANGED,41


,APP_ID,PREFIX,CLASS,SUFFIX,ARRAY_KEY,KEY,...,VALUE_BEFORE,VALUE_AFTER,ADJUSTED_BEFORE,ADJUSTED_AFTER,DIFF_FLAG,CHANGE_TYPE
3,I32554354,decisionResult,available_offers,,15210+manual,product_id,...,<NA>,15210,<NA>,(1K – 100K],1,ATTRIBUTE_NEW
4,I32554354,decisionResult,available_offers,,15210+standard,product_id,...,<NA>,15210,<NA>,(1K – 100K],1,ATTRIBUTE_NEW
5,I32554354,decisionResult,available_offers,,5123+standard,product_id,...,<NA>,5123,<NA>,(1K – 100K],1,ATTRIBUTE_NEW
6,I32554354,decisionResult,available_offers,,15210+manual,product_name,...,<NA>,KUPI_KUPI,<NA>,KUPI_KUPI,1,ATTRIBUTE_NEW
7,I32554354,decisionResult,available_offers,,15210+standard,product_name,...,<NA>,KUPI_KUPI,<NA>,KUPI_KUPI,1,ATTRIBUTE_NEW


## s8 · Качество данных

**Вход:** исходные документы и накопленная диагностика. **Выход:** `quality.parquet`.
Обязательные поля проверяются в исходных данных, независимо от фильтров сравнения.
Отказ по одному продукту должен иметь свою причину: код причины другого продукта
не скрывает пропуск. Ошибки чтения, отсутствующие пары, дубли и ограничения глубины
сохраняются в общей таблице.

In [11]:
show_source(steps.quality)
with stage("s8_quality", len(doc_pairs)) as log:
    quality_df = steps.quality(doc_pairs, cfg, issues)
    quality_df.to_parquet(run_dir / "quality.parquet", index=False)
    log["rows_out"] = len(quality_df)
preview(quality_df)
if quality_df.empty:
    print("Проблем качества текущими правилами не обнаружено.")

s8_quality: done; 0.0037 с; строк на выходе: 0
Строк: 0; колонок: 5
Проблем качества текущими правилами не обнаружено.


,APP_ID,ISSUE,SOURCE,PATH,DETAILS


## s9 · Разрезы по заявкам

**Выход:** `segments.parquet`, одна строка на заявку, включая повреждённые пары.
Здесь находятся переходы решений/причин, продукты, модели, пороги и число различий.
`COMPARISON_COMPLETE=0` означает, что отсутствие найденных различий не доказывает
равенство. `CAUSE=BOTH` фиксирует изменения входов и выходов; это не доказательство
того, что менялась стратегия. Модель/порог в одиночных полях — доминирующее значение;
полные множества сохраняются в дополнительных колонках.

In [12]:
show_source(steps.segments)
with stage("s9_segments", len(doc_pairs)) as log:
    segments_df = steps.segments(doc_pairs, comparison_df, cfg, quality_df)
    segments_df.to_parquet(run_dir / "segments.parquet", index=False)
    log["rows_out"] = len(segments_df)
assert len(segments_df) == len(pairs) and segments_df.APP_ID.is_unique
preview(segments_df[["APP_ID", "DECISION_TRANSITION", "REASON_TRANSITION", "N_DIFFS",
                     "CAUSE", "CAUSE_DETAIL", "QUALITY_ISSUE", "COMPARISON_COMPLETE"]])

s9_segments: done; 0.0142 с; строк на выходе: 4
Строк: 4; колонок: 8


,APP_ID,DECISION_TRANSITION,REASON_TRANSITION,N_DIFFS,CAUSE,CAUSE_DETAIL,QUALITY_ISSUE,COMPARISON_COMPLETE
0,I32554354,REJECT→APPROVE,AGELOW@21→∅,25,LOGIC,STOP_FACTOR,0,1
1,I32554355,REJECT→REJECT,MFO_ZAIMER_2026_MID→MFO_ZAIMER_2026_MID,0,NONE,NONE,0,1
2,I32554358,APPROVE→APPROVE,∅→∅,28,LOGIC,MODEL_CHANGE,0,1
3,I32554362,APPROVE→REJECT,∅→MFO_ZAIMER_2026_MID,30,BOTH,CUTOFF_CHANGE,0,1


## s10 · Массовые изменения по блокам

**Выход:** `blocks.parquet`, уровни иерархии и доля изменившихся заявок.
`APPS_TOTAL` — уникальные заявки с наблюдаемыми атрибутами внутри блока,
`APPS_CHANGED` — затронутые заявки в этом блоке. `SHARE` лежит в диапазоне 0–1.
Числа разных блоков не складываются: одна заявка может входить в несколько.

In [13]:
show_source(blocks)
with stage("s10_blocks", len(comparison_df)) as log:
    blocks_df = blocks(comparison_df, segments_df, cfg)
    blocks_df.to_parquet(run_dir / "blocks.parquet", index=False)
    log["rows_out"] = len(blocks_df)
preview(blocks_df.sort_values(["APPS_CHANGED", "BLOCK"], ascending=[False, True]), limit=8)

s10_blocks: done; 0.0237 с; строк на выходе: 18
Строк: 18; колонок: 14


,BLOCK,LEVEL,APPS_TOTAL,APPS_CHANGED,SHARE,ROWS_TOTAL,...,APPS_VALUE_CHANGED,APPS_ATTRIBUTE_NEW,APPS_ATTRIBUTE_LOST,ROWS_VALUE_CHANGED,ROWS_ATTRIBUTE_NEW,ROWS_ATTRIBUTE_LOST
2,decisionResult,1,4,3,0.750000,387,...,3,2,2,38,28,14
7,decisionResult.product_trials,2,4,3,0.750000,288,...,3,1,1,32,9,6
6,decisionResult.available_offers,2,3,2,0.666667,32,...,0,1,1,0,12,8
11,decisionResult.product_trials.cutOff,3,4,2,0.500000,18,...,2,0,0,10,0,0
13,decisionResult.product_trials.scoring,3,4,2,0.500000,81,...,2,1,0,10,9,0
16,decisionResult.product_trials.scoring.pd,4,4,2,0.500000,18,...,2,0,0,8,0,0
17,decisionResult.product_trials.scoring.scoreModels,4,4,2,0.500000,63,...,1,1,0,2,9,0
8,decisionResult.scoring,2,4,2,0.500000,63,...,1,1,0,4,7,0


## s11 · Факты для сводной и связи с заявками

**Выход:** `facts.parquet` и `fact_apps.parquet`.
В facts значение `CNT` равно количеству уникальных заявок при фиксированном
**полном наборе измерений**. Суммировать CNT по произвольным атрибутам нельзя.
Точные заявки связаны через `(FACT_ID, APP_ID)` в fact_apps.
Ниже показан локальный пример провала по FACT_ID; это проверка данных,
а не реализация REST или фронтенд-сводной.

In [14]:
show_source(facts)
with stage("s11_facts", len(comparison_df)) as log:
    facts_df, fact_apps_df = facts(comparison_df, segments_df, cfg)
    facts_df.to_parquet(run_dir / "facts.parquet", index=False)
    fact_apps_df.to_parquet(run_dir / "fact_apps.parquet", index=False)
    log["rows_out"] = len(facts_df)
assert not fact_apps_df.duplicated(["FACT_ID", "APP_ID"]).any()
counts = fact_apps_df.groupby("FACT_ID").APP_ID.nunique()
assert facts_df.set_index("FACT_ID").CNT.sort_index().tolist() == counts.sort_index().tolist()
preview(facts_df)
if not facts_df.empty:
    selected_fact = facts_df.sort_values(["DIFF_FLAG", "CNT"], ascending=False).iloc[0]
    selected_apps = fact_apps_df.loc[fact_apps_df.FACT_ID.eq(selected_fact.FACT_ID), "APP_ID"].tolist()
    assert len(selected_apps) == int(selected_fact.CNT)
    print("Проверка провала: FACT_ID =", int(selected_fact.FACT_ID), "CNT =", int(selected_fact.CNT))
    preview(segments_df.loc[segments_df.APP_ID.isin(selected_apps)])

s11_facts: done; 0.0277 с; строк на выходе: 609
Строк: 609; колонок: 21
Проверка провала: FACT_ID = 22 CNT = 1
Строк: 1; колонок: 35


,FACT_ID,PREFIX,CLASS,SUFFIX,KEY,PATH,...,REASON_AFTER,REASON_TRANSITION,CAUSE,PARTNER_TYPE,PRODUCTS,CNT
0,0,appData,application_id,,application_id,appData.application_id,...,<NA>,AGELOW@21→∅,LOGIC,CURRENT,DO_ZARPL;KUPI_KUPI,1
1,1,appData,application_id,,application_id,appData.application_id,...,<NA>,AGELOW@21→∅,LOGIC,CURRENT,DO_ZARPL;KUPI_KUPI,1
2,2,appData,application_id,,application_id,appData.application_id,...,MFO_ZAIMER_2026_MID,MFO_ZAIMER_2026_MID→MFO_ZAIMER_2026_MID,NONE,CURRENT,NA_KARTU;PODPISKA,1
3,3,appData,application_id,,application_id,appData.application_id,...,MFO_ZAIMER_2026_MID,MFO_ZAIMER_2026_MID→MFO_ZAIMER_2026_MID,NONE,CURRENT,NA_KARTU;PODPISKA,1
4,4,appData,application_id,,application_id,appData.application_id,...,<NA>,∅→∅,LOGIC,CURRENT,KUPI_KUPI;NA_KARTU;PODPISKA,1


,APP_ID,DECISION_BEFORE,DECISION_AFTER,DECISION_CHANGED,DECISION_TRANSITION,REASON_BEFORE,...,CHANGED_INPUT,CHANGED_LOGIC,PD_MODELS_BEFORE,PD_MODELS_AFTER,CUTOFFS_BEFORE,CUTOFFS_AFTER
3,I32554362,APPROVE,REJECT,1,APPROVE→REJECT,<NA>,...,1,1,ML_MICRO_LOAN_V2,ML_MICRO_LOAN_V2,MFO_ZAIMER_2026,MFO_ZAIMER_2026_MID


## s12 · Автоматические проверки

Вызывается **тот же `run_checks(state)`**, что и платформой: реальные CHECK-скрипты
через адаптер и отдельный процесс. `MartAPI` использует индексы по колонкам витрины.
`FAIL` — аналитическое срабатывание, а не падение Python; `ERROR` — ошибка исполнения.
В представлении ниже PASS/FAIL отображаются как ДА/НЕТ, LOW выделяется бледно;
исходные значения контракта не изменяются.

In [15]:
from engine.platform_checks_adapter import run_checks as adapter_run_checks
from engine.checks_service import run_checks as execute_checks
show_source(run_checks, adapter_run_checks, execute_checks, MartAPI)
with stage("s12_checks", len(comparison_df)) as log:
    checks_path = Path(run_checks({"cfg": cfg, "run_dir": str(run_dir)}))
    checks_result = json.loads(checks_path.read_text(encoding="utf-8"))
    log["rows_out"] = len(checks_result)
checks_df = pd.DataFrame(checks_result)
if checks_df.empty:
    print("Набор проверок пуст: проверьте cfg.checks.")
else:
    display(checks_df.RESULT.value_counts().rename("Строк отчёта").to_frame())
    view = checks_df[["CHECK_ID", "ARGUMENT", "CNT", "TOTAL", "METRIC", "RESULT", "CONFIDENCE"]].copy()
    view["RESULT"] = view.RESULT.replace({"PASS": "ДА", "FAIL": "НЕТ", "ERROR": "ОШИБКА"})
    headers = "".join("<th>" + html.escape(str(c)) + "</th>" for c in view.columns)
    rendered = []
    for _, row in view.head(20).iterrows():
        style = "opacity:0.55" if row.CONFIDENCE == "LOW" else ""
        rendered.append('<tr style="' + style + '">' +
                        "".join("<td>" + html.escape(str(v)) + "</td>" for v in row) + "</tr>")
    display(HTML("<table><thead><tr>" + headers + "</tr></thead><tbody>" + "".join(rendered) + "</tbody></table>"))

s12_checks: done; 0.3770 с; строк на выходе: 28


,Строк отчёта
RESULT,
FAIL,24
PASS,4


CHECK_ID,ARGUMENT,CNT,TOTAL,METRIC,RESULT,CONFIDENCE
ar_drop_by_product,DO_ZARPL,0,1,0.0,ДА,LOW
ar_drop_by_product,KUPI_KUPI,0,2,-1.0,ДА,LOW
ar_drop_by_product,NA_KARTU,1,3,0.5,НЕТ,LOW
ar_drop_by_product,NOVYY_MFO,1,1,1.0,НЕТ,LOW
ar_drop_by_product,PODPISKA,0,2,0.0,ДА,LOW
attribute_new_loss,decisionResult.available_offers.product_id,1,4,0.25,НЕТ,LOW
attribute_new_loss,decisionResult.available_offers.product_name,1,4,0.25,НЕТ,LOW
attribute_new_loss,decisionResult.available_offers.rank,1,4,0.25,НЕТ,LOW
attribute_new_loss,decisionResult.available_offers.type,1,4,0.25,НЕТ,LOW
attribute_new_loss,decisionResult.product_trials.rules.description,1,4,0.25,НЕТ,LOW


## s13 · Сводка и вердикт

**Выход:** `summary.json`, полученный тем же `prepare_summary`, что и сервером.
Доли одобрений и переходы решений считаются по заявкам с двумя известными решениями
и полным сравнением. Число строк различий и число затронутых заявок — разные метрики.
Вердикт показывает необходимость анализа, а не автоматически разрешает выпуск стратегии.

In [16]:
from engine.summary import build_summary
show_source(prepare_summary, build_summary)
with stage("s13_summary", len(segments_df)) as log:
    summary = prepare_summary(run_dir, cfg)
    log["rows_out"] = summary["total_apps"]
display(pd.DataFrame([{k: summary[k] for k in ["total_apps", "apps_with_diff", "apps_decision_changed",
                                              "apps_comparison_incomplete", "apps_with_quality_issues"]}]))
display(pd.DataFrame(summary["decision_transitions"]))
display(pd.DataFrame([summary["approval_rates"]]))
print("Вердикт:", summary["verdict"]["status"])

s13_summary: done; 0.0312 с; строк на выходе: 4
Вердикт: REVIEW_REQUIRED


,total_apps,apps_with_diff,apps_decision_changed,apps_comparison_incomplete,apps_with_quality_issues
0,4,3,2,0,0


,before,after,count,changed
0,APPROVE,APPROVE,1,False
1,APPROVE,REJECT,1,True
2,REJECT,APPROVE,1,True
3,REJECT,REJECT,1,False


,approve_value,denominator,before_count,after_count,before_rate,after_rate,delta_pp
0,APPROVE,4,2,2,0.5,0.5,0.0


## s14 · Необязательное мнение LLM

По умолчанию **пропущено**. Оно не входит в текущий `run_pipeline` и не нужно для
проверки воспроизводимости. Включение `RUN_LLM` вызывает существующий адаптер с
агрегированной сводкой, топом блоков и результатами проверок. В результатах проверок
могут быть APP_IDS. Ключ задаётся через окружение `LLM_API_KEY`, не в ноутбуке.
Если адаптер не вернул ответ, это явно показывается; аналитические результаты сохраняются.

In [17]:
llm_text = None
if RUN_LLM:
    from engine.llm import opinion
    show_source(opinion)
    with stage("s14_llm_optional") as log:
        llm_text = opinion(summary, blocks_df.sort_values("APPS_CHANGED", ascending=False).head(30).to_dict("records"), checks_result)
        log["rows_out"] = int(llm_text is not None)
    if llm_text:
        (run_dir / "llm_opinion.txt").write_text(llm_text, encoding="utf-8")
        display(Markdown(llm_text))
    else:
        print("Адаптер не вернул ответ: ключ, зависимость или API недоступны.")
else:
    print("s14: SKIPPED — необязательный сетевой вызов выключен.")

s14: SKIPPED — необязательный сетевой вызов выключен.


## s15 · Самостоятельный HTML-протокол

**Выход:** `report.html`, CSS и данные внутри. Сетевые ресурсы для чтения не нужны.
Ссылки внутри отчёта требуют работающего сервера. Ноутбук не регистрирует запуск
в SQLite платформы, поэтому открытие ссылки из локального notebook-run через API
может быть недоступно; это не мешает читать сам файл отчёта.

In [18]:
show_source(write_report_html)
with stage("s15_report") as log:
    report_path = write_report_html(run_dir, cfg)
    log["rows_out"] = 1
print("HTML-протокол:", display_path(report_path))
# Фрагмент HTML показываем как текст: сам файл откройте в браузере.
display(Code(report_path.read_text(encoding="utf-8")[:250], language="html"))

s15_report: done; 0.0012 с; строк на выходе: 1
HTML-протокол: runs/notebook_20261107_164621_5f6d09ff/report.html


<!doctype html>
<html lang="ru"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Протокол проверки — notebook_20261107_164621_5f6d09ff</title>
<style>
:root{--bg:#f6f8fb;--text:#203044;--border:#d

## Проверка: ноутбук = обычный пайплайн

При `VERIFY_PIPELINE=True` выполняется второй независимый запуск `run_pipeline`
на тех же исходных данных и конфиге. Сверяются **все семь Parquet-таблиц**, результаты
проверок и сводка. Исключаются только время исполнения и идентификатор запуска;
данные и аналитические метрики не исключаются. При расхождении ячейка останавливается
с ошибкой. Длительность этой проверки не включена во время пошагового прогона.

In [19]:
verification = {"status": "SKIPPED", "reason": "VERIFY_PIPELINE=False"}
if VERIFY_PIPELINE:
    verification_started = perf_counter()
    reference_dir = output_root / (run_id + "_reference")
    if reference_dir.exists():
        raise FileExistsError("Для новой сверки перезапустите ядро и выполните ноутбук сверху вниз.")
    reference_events = []
    reference_result = run_pipeline(deepcopy(cfg), reference_dir, on_event=reference_events.append)
    table_names = ["union", "comparison", "quality", "segments", "blocks", "facts", "fact_apps"]
    for name in table_names:
        actual = pd.read_parquet(run_dir / f"{name}.parquet")
        expected = pd.read_parquet(reference_dir / f"{name}.parquet")
        pd.testing.assert_frame_equal(actual, expected, check_exact=True)
    def check_values(rows):
        return [{k: v for k, v in row.items() if k not in {"DURATION_MS", "TRACEBACK"}} for row in rows]
    reference_checks = json.loads((reference_dir / "checks.json").read_text(encoding="utf-8"))
    assert check_values(checks_result) == check_values(reference_checks), "Различаются результаты CHECK"
    def summary_values(value):
        return {k: v for k, v in value.items() if k not in {"run_id", "summary_seconds"}}
    assert summary_values(summary) == summary_values(reference_result["summary"]), "Различаются сводки"
    verification = {"status": "PASS", "tables": table_names,
                    "checks_equal": True, "summary_equal": True,
                    "reference_run": reference_dir.name,
                    "duration_seconds": round(perf_counter() - verification_started, 4)}
    print("PASS: 7 таблиц, результаты проверок и сводка совпадают с run_pipeline.")
    print("Время отдельной сверки:", verification["duration_seconds"], "с")
else:
    print("Сверка с обычным пайплайном пропущена пользователем.")
save_json("notebook_verification.json", verification)

PASS: 7 таблиц, результаты проверок и сводка совпадают с run_pipeline.
Время отдельной сверки: 0.571 с


## Журнал и итоговые артефакты

Время ниже — вызов функции и запись артефакта внутри соответствующей ячейки.
В него не входят чтение человеком, отображение таблиц, подготовка конфигурации и
повторная сверка. Это измерения текущего компьютера и текущего набора данных.
Четыре демонстрационные пары не подтверждают целевую скорость на полном датасете.
Стадии s2–s4 разделены для объяснения; в платформенном журнале они объединены.

In [20]:
timings_df = pd.DataFrame(TIMINGS)
display(timings_df[["step", "status", "rows_in", "rows_out", "duration_seconds"]])
save_json("notebook_stage_log.json", TIMINGS)
execution = {"mode": "demo" if using_demo else "dataset", "run_id": run_id,
             "total_apps": len(pairs), "calculation_seconds": round(timings_df.duration_seconds.sum(), 4),
             "verification": verification["status"]}
save_json("notebook_execution.json", execution)
print("Сумма времени стадий:", execution["calculation_seconds"], "с")
print("Режим:", execution["mode"], "· Заявок:", execution["total_apps"])
preview(pd.DataFrame([{"Файл": p.name, "Размер, байт": p.stat().st_size}
                      for p in sorted(run_dir.iterdir()) if p.is_file()]), limit=20)
print("Все результаты:", display_path(run_dir))

Сумма времени стадий: 0.5607 с
Режим: demo · Заявок: 4
Строк: 15; колонок: 2
Все результаты: runs/notebook_20261107_164621_5f6d09ff


,step,status,rows_in,rows_out,duration_seconds
0,s1_discover,done,NaN,4,0.0002
1,s2_load,done,4.0,4,0.0004
2,s3_flatten,done,4.0,1148,0.0372
3,s4_bucketize,done,1148.0,1148,0.0018
4,s5_union,done,1148.0,1148,0.0191
...,...,...,...,...,...
9,s10_blocks,done,595.0,18,0.0237
10,s11_facts,done,595.0,609,0.0277
11,s12_checks,done,595.0,28,0.3770
12,s13_summary,done,4.0,4,0.0312


,Файл,"Размер, байт"
0,blocks.parquet,9703
1,checks.json,9834
2,checks_log.json,918
3,comparison.parquet,17486
4,comparison_status.json,248
...,...,...
10,quality.parquet,2635
11,report.html,13825
12,segments.parquet,22273
13,summary.json,8852


## Что показать на защите

1. Источник и конфиг: режим demo/полный датасет, исключения, ключи массивов.
2. s3 и s7: `PATH + ARRAY_KEY`, до/после, отдельные NEW/LOST.
3. s8–s9: проблемы качества, переходы решений и отличие входов от результатов.
4. s10–s11: блоки, distinct CNT, точные заявки через fact_apps.
5. s12–s13: пять реальных проверок, FAIL vs ERROR, вердикт и матрица решений.
6. Исходный код любой стадии, HTML и итог независимой сверки.

**Границы этого шага:** UI pivot/drill-down, Docker, пользовательский редактор и
журнал FastAPI проверяются владельцами платформы и фронта. Ноутбук использует
текущий последовательный движок; параллельный парсинг этим шагом не добавлялся.